> **Prefer Google Colab?** [Open this notebook in Colab](https://colab.research.google.com/github/SpatialTurn/DataCollection/blob/main/notebooks/01b_Census_API_Basics.ipynb).
> In Colab, click **File > Save a copy in Drive** first so your changes are kept.

# Getting Data from the Census API

**Series:** Census Data in Python (notebook 2 of 3)
**Time:** about 35 minutes

**In this notebook you will:**
- Understand the parts of a Census API request
- Look up what a variable code means, and search for variables
- Download American Community Survey (ACS) estimates for counties and census tracts
- Use margins of error to judge how reliable an estimate is
- Compare the ACS with the 2020 Decennial Census

**You should already know:** Notebook 1a (Census Boundaries). You also need a free Census API key.

> **Before you start:** run each cell in order by clicking it and pressing **Shift + Enter**.

## Background: what is an API?

An **API** (Application Programming Interface) lets a program ask a server for data using a web
address, instead of a person clicking through a website. The Census Data API answers with a table.
Every request has the same parts:

```
https://api.census.gov/data/2024/acs/acs5 ? get=NAME,B01003_001E & for=county:* & in=state:18 & key=...
\_______________________/\____________/   \________________/   \_________/   \________/   \_____/
        the API             the dataset      the variables       geography   inside what   your key
```

| Part | Example | Meaning |
|---|---|---|
| Dataset | `2024/acs/acs5` | ACS 5-year estimates ending in 2024 (covering 2020 to 2024) |
| `get` | `NAME,B01003_001E` | Which columns you want, separated by commas |
| `for` | `county:*` | Which geography; `*` means all of them |
| `in` | `state:18` | The larger area they sit inside |
| `key` | your key | Identifies you; required for every request |

The API only ever returns **summary (aggregate) data** for areas. It never returns information about
individual people.

## Your Census API key

> **Every learner needs their own free Census API key.** Request one at
> <https://api.census.gov/data/key_signup.html>. It arrives by email within a few minutes, and you
> must click the activation link in that email before it works.

Paste your key between the quotes in the next cell and run it. Treat the key like a password: do not
post it online or leave it in a notebook you share. If you share this notebook, delete the key first.

In [ ]:
CENSUS_API_KEY = "PASTE_YOUR_CENSUS_API_KEY_HERE"

if CENSUS_API_KEY.startswith("PASTE"):
    raise ValueError("Paste your free Census API key between the quotes above, then run this cell again.")
print("Key saved. It is", len(CENSUS_API_KEY), "characters long (a valid key has 40).")

## Step 1: Import libraries

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

## Step 2: Your first request

Let us ask for the total population of every state. `requests.get` sends the request; we pass the
parts of the address as a dictionary called `params`, and `requests` builds the address for us.

In [ ]:
reply = requests.get(
    "https://api.census.gov/data/2024/acs/acs5",
    params={"get": "NAME,B01003_001E", "for": "state:*", "key": CENSUS_API_KEY},
    timeout=60,
)
print("Status code:", reply.status_code)      # 200 means success
rows = reply.json()
rows[:4]

The reply is a **list of lists**. The first list is the header row; each list after it is one state.
Notice that the numbers arrive as text (`'5157687'` in quotes). Let us turn it into a DataFrame and
convert the numbers.

In [ ]:
pop = pd.DataFrame(rows[1:], columns=rows[0])
pop["B01003_001E"] = pd.to_numeric(pop["B01003_001E"])
pop = pop.rename(columns={"B01003_001E": "total_pop"})
pop.sort_values("total_pop", ascending=False).head()

## Step 3: A helper function

We will make many requests, so the next cell wraps those steps in a function, `census_get`. It also
checks for problems: when something is wrong (a missing key, a misspelled variable), the API sends back
a web page or a message instead of a table, and the function shows you the start of that message.

In [ ]:
import requests

def census_get(dataset, variables, geo_for, geo_in=None):
    """Ask the Census API for some variables and return a pandas DataFrame.

    dataset   : the part of the address after /data/, for example "2024/acs/acs5"
    variables : a list of variable codes, for example ["NAME", "B01003_001E"]
    geo_for   : the geography you want, for example "county:*" (all counties)
    geo_in    : the geography it sits inside, for example "state:18" (Indiana)
    """
    params = {"get": ",".join(variables), "for": geo_for, "key": CENSUS_API_KEY}
    if geo_in:
        params["in"] = geo_in
    reply = requests.get(f"https://api.census.gov/data/{dataset}", params=params, timeout=60)

    # The API answers with JSON when it works, and with a web page or a short message when it does not
    if reply.status_code != 200 or "json" not in reply.headers.get("Content-Type", ""):
        raise RuntimeError(
            f"The Census API did not return data (status {reply.status_code}). "
            f"The start of its reply was:\n{reply.text[:300]}"
        )

    rows = reply.json()                                  # a list of lists; the first list is the header
    table = pd.DataFrame(rows[1:], columns=rows[0])
    for v in variables:                                  # the API sends every value as text
        if v != "NAME":
            table[v] = pd.to_numeric(table[v], errors="coerce")
    return table

In [ ]:
census_get("2024/acs/acs5", ["NAME", "B01003_001E"], "county:*", "state:18").head()

## Step 4: What do the variable codes mean?

ACS variables follow a pattern:

| Part | Example | Meaning |
|---|---|---|
| Table type | `B` | **B**: detailed tables (most variables). **DP**: data profiles (ready-made percentages). **S**: subject tables |
| Table and line | `B01003_001` | Table B01003 (Total Population), line 001 |
| Suffix | `E` | **E**: estimate. **M**: margin of error. **PE** and **PM**: percent estimate and its margin of error (profiles only) |

The data profiles live in a slightly different dataset address: `2024/acs/acs5/profile`.

Every variable's description is available from the API too, and looking one up does not need a key:

In [ ]:
def describe(dataset, variable):
    info = requests.get(f"https://api.census.gov/data/{dataset}/variables/{variable}.json", timeout=60).json()
    return info["label"].replace("!!", " > ")

for v in ["B01003_001E", "B19013_001E", "B08201_002E"]:
    print(v, ":", describe("2024/acs/acs5", v))
print("DP04_0058PE :", describe("2024/acs/acs5/profile", "DP04_0058PE"))

To **search** for a variable, download the list of all variables in a dataset and look for words in
their labels. This is the same list you can browse in a web browser at
<https://api.census.gov/data/2024/acs/acs5/profile/variables.html>.

In [ ]:
catalog = requests.get("https://api.census.gov/data/2024/acs/acs5/profile/variables.json", timeout=120).json()
variables = pd.DataFrame(
    [(name, v.get("label", ""), v.get("concept", "")) for name, v in catalog["variables"].items()],
    columns=["code", "label", "concept"],
)
print(len(variables), "variables in the data profiles")

matches = variables[variables["label"].str.contains("No vehicles available", case=False)]
matches.sort_values("code")

## Step 5: Households without a car, by county

Not having a car makes it much harder to reach a grocery store, a doctor or a job. The data profile
variable `DP04_0058` counts occupied housing units (households) with no vehicle. We ask for the count,
the percent, and the margin of error of each.

In [ ]:
cars = census_get(
    "2024/acs/acs5/profile",
    ["NAME", "DP04_0058E", "DP04_0058M", "DP04_0058PE", "DP04_0058PM"],
    "county:*", "state:18",
)
cars = cars.rename(columns={
    "DP04_0058E": "no_car", "DP04_0058M": "no_car_moe",
    "DP04_0058PE": "pct_no_car", "DP04_0058PM": "pct_no_car_moe",
})
cars["NAME"] = cars["NAME"].str.replace(" County, Indiana", "")
cars.sort_values("no_car", ascending=False).head()

By **count**, the counties with the most car-free households are the biggest ones, led by Marion County
(Indianapolis). That mostly tells us where people live. Sorting by **percent** tells a different story:

In [ ]:
cars.sort_values("pct_no_car", ascending=False).head()

LaGrange County, in northern Indiana, has by far the highest share of households without a vehicle.
It is home to one of the largest Amish communities in the United States, and many Amish households
do not own cars. This is why maps and comparisons should usually use **rates or percentages**, not
raw counts (see the Data Visualization module on normalization).

## Step 6: Margins of error

The ACS is a **survey** of a sample of households, so every number is an **estimate** with a **margin
of error** (MOE). The Census Bureau publishes 90% margins of error: we are 90% confident the true value
is within the estimate plus or minus the MOE.

A common way to judge reliability is the **coefficient of variation** (CV): the standard error as a
percentage of the estimate. Because a 90% MOE is 1.645 standard errors:

`CV = (MOE / 1.645) / estimate * 100`

A widely used rule of thumb: CV below 12% is high reliability, 12% to 40% is medium, and above 40% is
low reliability.

In [ ]:
cars["low"] = cars["no_car"] - cars["no_car_moe"]
cars["high"] = cars["no_car"] + cars["no_car_moe"]
cars["cv"] = (cars["no_car_moe"] / 1.645) / cars["no_car"] * 100
cars["reliability"] = pd.cut(cars["cv"], [0, 12, 40, float("inf")], labels=["high", "medium", "low"])

cars[["NAME", "no_car", "low", "high", "cv", "reliability"]].sort_values("cv", ascending=False).head()

In [ ]:
cars["reliability"].value_counts()

Small counties have fewer survey responses, so their estimates are less certain. Always look at the
margins of error before drawing conclusions about small areas, and say so when you report them.

## Step 7: Tract-level data and special values

For census tracts, ask `for` `tract:*` and put both the state and county in `in`, separated by a space.
Here we get median household income (`B19013_001E`) for the tracts of Tippecanoe County.

In [ ]:
income = census_get(
    "2024/acs/acs5",
    ["NAME", "B01003_001E", "B19013_001E", "B19013_001M"],
    "tract:*", "state:18 county:157",
)
income = income.rename(columns={"B01003_001E": "total_pop", "B19013_001E": "median_income",
                                "B19013_001M": "median_income_moe"})
print(len(income), "tracts")
income.sort_values("median_income").head()

Some values are large negative numbers such as `-666666666`. These are **annotation codes**, not real
values. `-666666666` means the estimate could not be calculated, for example because too few households
were sampled. That happens in tracts where most residents live in **group quarters** such as university
residence halls, which are not households. Always replace these codes with "missing" before doing math:

In [ ]:
import numpy as np

income = income.replace([-666666666, -999999999, -888888888, -555555555, -333333333, -222222222], np.nan)
print("Tracts without a median income:", income["median_income"].isna().sum())
income["median_income"].describe()

## Step 8: The 2020 Decennial Census

The **Decennial Census** counts every person in the country every ten years. Because it is a count, not
a sample, it has no margin of error, but it asks only a few questions (age, sex, race, Hispanic origin,
household relationship and housing tenure). The 2020 Demographic Profile is dataset `2020/dec/dp`.
`DP1_0024P` is the percent of people aged 65 and over.

In [ ]:
older = census_get("2020/dec/dp", ["NAME", "DP1_0001C", "DP1_0024C", "DP1_0024P"], "county:*", "state:18")
older = older.rename(columns={"DP1_0001C": "total_pop", "DP1_0024C": "age65plus", "DP1_0024P": "pct_65plus"})
older["NAME"] = older["NAME"].str.replace(" County, Indiana", "")
older.sort_values("pct_65plus", ascending=False)

Tippecanoe County has the youngest population in Indiana, because of its large student population.

| | American Community Survey | Decennial Census |
|---|---|---|
| How | Sample of about 3.5 million addresses a year | Count of everyone |
| How often | Every year (1-year and 5-year estimates) | Every 10 years |
| Topics | Income, education, commuting, vehicles, internet, language and much more | Age, sex, race, Hispanic origin, households, housing tenure |
| Margin of error | Yes | No |
| Smallest areas | Block groups (5-year estimates) | Blocks |

## Step 9: Two handy options

Add `descriptive=true` to get a second header row with plain-language labels, or
`outputFormat=csv` to get a CSV file instead of JSON. A CSV reply can go straight into pandas:

In [ ]:
from io import StringIO

reply = requests.get(
    "https://api.census.gov/data/2024/acs/acs5/profile",
    params={"get": "NAME,DP04_0058E,DP04_0058PE", "for": "county:157", "in": "state:18",
            "outputFormat": "csv", "key": CENSUS_API_KEY},
    timeout=60,
)
pd.read_csv(StringIO(reply.text))

## Try it yourself

Get **median household income** (`B19013_001E`) and its margin of error (`B19013_001M`) for every
county in Indiana. Which county has the highest median household income?

*Hint:* Use `census_get` with the dataset `2024/acs/acs5`, then `sort_values`.

<details>
<summary><b>Show a solution</b></summary>

```python
inc = census_get("2024/acs/acs5", ["NAME", "B19013_001E", "B19013_001M"], "county:*", "state:18")
inc.sort_values("B19013_001E", ascending=False).head(3)
```

</details>

In [ ]:
# Write your answer here

## Recap

- A Census API request names a dataset, the variables (`get`), the geography (`for` and `in`) and your key
- Variable codes end in E (estimate) or M (margin of error); data profiles add PE and PM for percents
- Compare places with percentages, not counts
- ACS numbers are estimates: check margins of error, especially for small areas
- Replace annotation codes such as -666666666 with missing values before calculating

**Next:** [Joining Census Tables to Maps](01c_Joining_Tables_to_Maps.ipynb)